# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **DOI/Identifier:** 10.71728/senscience.vq4a-28xa
- **Description:**
    This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant pandas --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('--- Dataset Metadata ---')
print(f"Title: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description[:350]}...")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

**Note:** All entities below are referenced by their `@id` fields for consistency and clarity.


In [ ]:
# List available record sets and, for each, its fields and columns (@id)

if not metadata.record_sets or len(metadata.record_sets) == 0:
    print("No record sets found via metadata.record_sets; trying metadata.record_set (legacy Croissant schema field)...")

record_sets = getattr(metadata, 'record_sets', None)
if not record_sets:
    # Some schemas use record_set (not plural) as a list
    record_sets = getattr(metadata, 'record_set', [])

set_ids = []
for rset in record_sets:
    print(f"\nRecordSet @id: {rset.id}")
    set_ids.append(rset.id)
    print(f"  Name: {getattr(rset, 'name', 'N/A')}")
    print(f"  Description: {getattr(rset, 'description', 'N/A')}")
    # Fields
    print("  Fields:")
    if hasattr(rset, 'fields') and rset.fields:
        for field in rset.fields:
            print(f"    - @id: {field.id}\tName: {getattr(field, 'name', '')}")
    # Columns
    if hasattr(rset, 'columns') and rset.columns:
        print("  Columns:")
        for col in rset.columns:
            print(f"    - @id: {col.id}\tName: {getattr(col, 'name', col.id)}")

if not set_ids:
    print("No record sets available in this Croissant schema.")
else:
    print(f"\nRecord set IDs found: {set_ids}")

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Use the record set and field `@id` from the previous overview. All entities are referenced using their `@id`.


In [ ]:
# You may need to adapt the next block if your dataset
# contains multiple record sets or hierarchical structures.
dataframes = {}

print('Loading all available record sets...')

for record_set_id in set_ids:
    print(f"- Loading record set: {record_set_id}")
    records = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(list(records))
    dataframes[record_set_id] = df
    print(f"  DataFrame shape: {df.shape}")
    if len(df.columns) > 0:
        print(f"  Columns: {df.columns.tolist()[:6]}{'...' if len(df.columns) > 6 else ''}")

if len(dataframes) == 0:
    print('No dataframes created. Check if the dataset has public record sets.')
else:
    # Preview the first record set with actual data
    for rid, rdf in dataframes.items():
        print(f"\n--- Preview of record set {rid} ---")
        display(rdf.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All references are by `@id` (`record_set_id`, `field_id`).


In [ ]:
# Choose your record set to analyze (pick the first one unless you know a better one)
if len(dataframes) == 0:
    raise ValueError('No data loaded; cannot perform EDA.')
record_set_id = set_ids[0]
df = dataframes[record_set_id]
# Show all columns for reference
print('Available columns:', list(df.columns))

# Choose a numeric field (by column name, which is typically the field @id, e.g. 'cr:peptide_count')
# Here, we try to autodetect a numeric column:
numeric_field_candidates = df.select_dtypes(include=[int, float]).columns.tolist()
if not numeric_field_candidates:
    print('No numeric columns auto-detected. Displaying first few rows for manual inspection:')
    display(df.head())
    # You may need to manually select below.
    numeric_field = None
else:
    numeric_field = numeric_field_candidates[0]  # Use the first detected numeric column
    print(f'Using numeric field: {numeric_field}')

if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
    print(f"Normalized {numeric_field} (z-score):")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try group by a string/categorical column if available
    group_field_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
    group_field = None
    if group_field_candidates:
        # Skip any field which is likely an id, prefer those with word-like names
        group_field = [x for x in group_field_candidates if ('name' in x.lower() or 'type' in x.lower() or 'label' in x.lower())]
        # Otherwise just take first string field
        group_field = group_field[0] if group_field else group_field_candidates[0]

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print('No numeric field selected; skipping filtering and normalization.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we provide sample code for plotting histograms and scatter plots for the first numeric field detected.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If there is another numeric field, show a scatter plot
    if len(numeric_field_candidates) >= 2:
        plt.figure(figsize=(7, 4))
        x_field, y_field = numeric_field_candidates[:2]
        sns.scatterplot(x=df[x_field], y=df[y_field])
        plt.xlabel(x_field)
        plt.ylabel(y_field)
        plt.title(f"Scatter plot of {x_field} vs {y_field}")
        plt.show()

else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated loading a Croissant-based mass spectrometry protein dataset via `mlcroissant`, explored its available record sets and fields by `@id`, loaded the data into pandas DataFrames, and performed basic EDA including filtering, normalization, grouping, and visualization. For advanced use, you may proceed to train or evaluate models on the extracted data, or further clean/engineer its features based on research needs.

- Please consult the [FAIR² data citation](https://doi.org/10.71728/senscience.vq4a-28xa) if you use this dataset.
- For more advanced queries, always use entity `@id`s as the preferred reference for fields/columns in `mlcroissant`.